## MULTIPLE VALIDATION versus CONSENSUS

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, accuracy_score

In [ ]:
EXCEL_PATH = "../review_documents/PECARN MULTIPLE VALIDATION.xlsx"

PAIRS = [
    ("TBI OLDER LEISHA", "TBI OLDER"),
    ("TBI YOUNGER LEISHA", "TBI YOUNGER"),
    ("CSPINE LEISHA", "CSPINE"),
    ("IAI LEISHA", "IAI"),
]

In [ ]:
xl = pd.ExcelFile(EXCEL_PATH)

def normalize_col(series):
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().all():
        return numeric.astype(int).astype(str)
    return series.astype(str)

for leisha_sheet, kush_sheet in PAIRS:
    df_leisha = pd.read_excel(xl, sheet_name=leisha_sheet, header=1)
    df_kush = pd.read_excel(xl, sheet_name=kush_sheet, header=1)

    if "exclude" in df_leisha.columns:
        df_leisha = df_leisha[df_leisha["exclude"] != "yes"]
    if "exclude" in df_kush.columns:
        df_kush = df_kush[df_kush["exclude"] != "yes"]

    risk_cols = [c for c in df_leisha.columns if c.startswith("risk_")]

    merged = df_leisha[["EncounterKey"] + risk_cols].merge(
        df_kush[["EncounterKey"] + risk_cols],
        on="EncounterKey",
        suffixes=("_leisha", "_kush"),
    )

    results = []
    for col in risk_cols:
        col_l = f"{col}_leisha"
        col_k = f"{col}_kush"
        valid = merged[[col_l, col_k]].copy()
        valid = valid.replace(".", pd.NA).dropna()
        if len(valid) == 0:
            continue
        valid[col_l] = normalize_col(valid[col_l])
        valid[col_k] = normalize_col(valid[col_k])
        kappa = cohen_kappa_score(valid[col_l], valid[col_k])
        acc = accuracy_score(valid[col_l], valid[col_k])
        results.append({"variable": col, "n": len(valid), "kappa": round(kappa, 3), "accuracy": round(acc, 3)})

    print(f"\n=== {kush_sheet} (n={len(merged)}) ===")
    display(pd.DataFrame(results))

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

OUTPUT_PATH = "../review_documents/PECARN HUMAN CONSENSUS_BAD.xlsx"
RED_FILL = PatternFill(start_color="FF0000", end_color="FF0000", fill_type="solid")

SHEET_NAMES = {
    "TBI OLDER": "TBI OLDER HUMAN CONSENSUS",
    "TBI YOUNGER": "TBI YOUNGER HUMAN CONSENSUS",
    "CSPINE": "CSPINE HUMAN CONSENSUS",
    "IAI": "IAI HUMAN CONSENSUS",
}

def normalize_val(val):
    if pd.isna(val) or val == ".":
        return None
    try:
        return str(int(float(val)))
    except (ValueError, TypeError):
        return str(val)

disagreement_map = {}

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    for leisha_sheet, kush_sheet in PAIRS:
        df_leisha = pd.read_excel(xl, sheet_name=leisha_sheet, header=1)
        df_kush = pd.read_excel(xl, sheet_name=kush_sheet, header=1)

        if "exclude" in df_leisha.columns:
            df_leisha = df_leisha[df_leisha["exclude"] != "yes"]
        if "exclude" in df_kush.columns:
            df_kush = df_kush[df_kush["exclude"] != "yes"]

        risk_cols = [c for c in df_kush.columns if c.startswith("risk_")]
        non_risk_cols = [c for c in df_kush.columns if not c.startswith("risk_")]

        merged = df_kush[non_risk_cols + risk_cols].merge(
            df_leisha[["EncounterKey"] + risk_cols],
            on="EncounterKey",
            suffixes=("_kush", "_leisha"),
        )

        output = merged[non_risk_cols].copy()
        discordant_cells = []

        for col in risk_cols:
            col_k = f"{col}_kush"
            col_l = f"{col}_leisha"
            vals = []
            for idx, row in merged.iterrows():
                v_k = normalize_val(row[col_k])
                v_l = normalize_val(row[col_l])
                if v_k == v_l:
                    vals.append(v_k if v_k is not None else "")
                else:
                    label = f"LEISHA: {row[col_l]} - KUSH: {row[col_k]}"
                    vals.append(label)
                    discordant_cells.append((len(vals) - 1, col))
            output[col] = vals

        sheet_name = SHEET_NAMES[kush_sheet]
        output.to_excel(writer, sheet_name=sheet_name, index=False)
        disagreement_map[sheet_name] = [
            (r, output.columns.get_loc(c)) for r, c in discordant_cells
        ]

wb = load_workbook(OUTPUT_PATH)
for sheet_name, cells in disagreement_map.items():
    ws = wb[sheet_name]
    for row_idx, col_idx in cells:
        ws.cell(row=row_idx + 2, column=col_idx + 1).fill = RED_FILL

wb.save(OUTPUT_PATH)
print(f"Saved to {OUTPUT_PATH}")
for sheet_name, cells in disagreement_map.items():
    print(f"  {sheet_name}: {len(cells)} disagreement cells")

In [ ]:
wb_check = load_workbook(OUTPUT_PATH)
checks_passed = 0
checks_total = 0

for leisha_sheet, kush_sheet in PAIRS:
    sheet_name = SHEET_NAMES[kush_sheet]
    ws = wb_check[sheet_name]

    df_leisha = pd.read_excel(xl, sheet_name=leisha_sheet, header=1)
    df_kush = pd.read_excel(xl, sheet_name=kush_sheet, header=1)
    if "exclude" in df_leisha.columns:
        df_leisha = df_leisha[df_leisha["exclude"] != "yes"]
    if "exclude" in df_kush.columns:
        df_kush = df_kush[df_kush["exclude"] != "yes"]

    risk_cols = [c for c in df_kush.columns if c.startswith("risk_")]
    merged = df_kush.merge(
        df_leisha[["EncounterKey"] + risk_cols],
        on="EncounterKey",
        suffixes=("_kush", "_leisha"),
    )

    headers = [ws.cell(row=1, column=c).value for c in range(1, ws.max_column + 1)]
    enc_col_idx = headers.index("EncounterKey") + 1

    encounter_rows = {}
    for row in range(2, ws.max_row + 1):
        encounter_rows[ws.cell(row=row, column=enc_col_idx).value] = row

    sample_size = min(3, len(merged))
    for _, mrow in merged.sample(sample_size, random_state=42).iterrows():
        enc_key = mrow["EncounterKey"]
        excel_row = encounter_rows[enc_key]
        for col in risk_cols:
            v_k = normalize_val(mrow[f"{col}_kush"])
            v_l = normalize_val(mrow[f"{col}_leisha"])
            col_idx = headers.index(col) + 1
            cell = ws.cell(row=excel_row, column=col_idx)
            cell_val = str(cell.value) if cell.value is not None else ""
            has_red = cell.fill.start_color.rgb == "00FF0000"

            checks_total += 1
            if v_k == v_l:
                ok = not has_red and "LEISHA:" not in cell_val
                if not ok:
                    print(f"FAIL [agree] {sheet_name} | {enc_key} | {col}: val={cell_val}, red={has_red}")
                else:
                    checks_passed += 1
            else:
                ok = has_red and "LEISHA:" in cell_val and "KUSH:" in cell_val
                if not ok:
                    print(f"FAIL [disagree] {sheet_name} | {enc_key} | {col}: val={cell_val}, red={has_red}")
                else:
                    checks_passed += 1

print(f"\n{checks_passed}/{checks_total} spot checks passed")

In [ ]:
total_cells = 0
total_disagreements = 0
rows_data = []

for leisha_sheet, kush_sheet in PAIRS:
    df_leisha = pd.read_excel(xl, sheet_name=leisha_sheet, header=1)
    df_kush = pd.read_excel(xl, sheet_name=kush_sheet, header=1)

    if "exclude" in df_leisha.columns:
        df_leisha = df_leisha[df_leisha["exclude"] != "yes"]
    if "exclude" in df_kush.columns:
        df_kush = df_kush[df_kush["exclude"] != "yes"]

    risk_cols = [c for c in df_kush.columns if c.startswith("risk_")]
    merged = df_kush[["EncounterKey"] + risk_cols].merge(
        df_leisha[["EncounterKey"] + risk_cols],
        on="EncounterKey",
        suffixes=("_kush", "_leisha"),
    )

    n_cells = 0
    n_disagree = 0
    for col in risk_cols:
        valid = merged[[f"{col}_kush", f"{col}_leisha"]].copy()
        valid = valid.replace(".", pd.NA).dropna()
        n_cells += len(valid)
        disagree = sum(
            normalize_val(row[f"{col}_kush"]) != normalize_val(row[f"{col}_leisha"])
            for _, row in valid.iterrows()
        )
        n_disagree += disagree

    total_cells += n_cells
    total_disagreements += n_disagree
    rows_data.append({
        "dataset": kush_sheet,
        "rated_cells": n_cells,
        "disagreements": n_disagree,
        "proportion": round(n_disagree / n_cells, 3),
    })

rows_data.append({
    "dataset": "TOTAL",
    "rated_cells": total_cells,
    "disagreements": total_disagreements,
    "proportion": round(total_disagreements / total_cells, 3),
})

display(pd.DataFrame(rows_data))

## HUMAN CONSENSUS error types

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles.colors import COLOR_INDEX
from collections import Counter
import xml.etree.ElementTree as ET

file_path = "../review_documents/PECARN HUMAN CONSENSUS.xlsx"
wb = load_workbook(file_path)

THEME_ORDER = [
    "dk1", "lt1", "dk2", "lt2",
    "accent1", "accent2", "accent3", "accent4", "accent5", "accent6",
    "hlink", "folHlink"
]

def apply_tint(rgb, tint):
    if tint is None or tint == 0:
        return rgb.upper()

    r = int(rgb[0:2], 16)
    g = int(rgb[2:4], 16)
    b = int(rgb[4:6], 16)

    def tint_channel(c, tint_value):
        if tint_value < 0:
            return int(round(c * (1.0 + tint_value)))
        return int(round(c * (1.0 - tint_value) + 255 * tint_value))

    r = max(0, min(255, tint_channel(r, tint)))
    g = max(0, min(255, tint_channel(g, tint)))
    b = max(0, min(255, tint_channel(b, tint)))

    return f"{r:02X}{g:02X}{b:02X}"

def get_theme_map(workbook):
    theme_map = {}
    if not getattr(workbook, "loaded_theme", None):
        return theme_map

    root = ET.fromstring(workbook.loaded_theme)
    ns = {"a": "http://schemas.openxmlformats.org/drawingml/2006/main"}

    clr_scheme = root.find(".//a:clrScheme", ns)
    if clr_scheme is None:
        return theme_map

    for i, key in enumerate(THEME_ORDER):
        elem = clr_scheme.find(f"a:{key}", ns)
        if elem is None:
            continue

        child = list(elem)[0] if list(elem) else None
        if child is None:
            continue

        if child.tag.endswith("sysClr"):
            rgb = child.attrib.get("lastClr")
        else:
            rgb = child.attrib.get("val")

        if rgb:
            theme_map[i] = rgb.upper()

    return theme_map

THEME_MAP = get_theme_map(wb)

def resolve_color(color_obj):
    if color_obj is None:
        return None

    ctype = color_obj.type

    if ctype == "rgb" and color_obj.rgb:
        rgb = color_obj.rgb[-6:]
        return f"#{rgb.upper()}"

    if ctype == "indexed" and color_obj.indexed is not None:
        idx = int(color_obj.indexed)
        if 0 <= idx < len(COLOR_INDEX):
            rgb = COLOR_INDEX[idx]
            if rgb and len(rgb) >= 6:
                return f"#{rgb[-6:].upper()}"
        return f"indexed:{idx}"

    if ctype == "theme" and color_obj.theme is not None:
        theme_idx = int(color_obj.theme)
        base_rgb = THEME_MAP.get(theme_idx)
        if base_rgb:
            tint = getattr(color_obj, "tint", 0) or 0
            rgb = apply_tint(base_rgb, tint)
            return f"#{rgb.upper()}"
        return f"theme:{theme_idx}"

    if ctype == "auto":
        return "auto"

    return None

def get_fill_color(cell):
    fill = cell.fill
    if fill is None or fill.fill_type != "solid":
        return None
    return resolve_color(fill.fgColor)

sheet_counts = {}
cumulative = Counter()

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    counts = Counter()

    for row in ws.iter_rows():
        for cell in row:
            color = get_fill_color(cell)
            if color:
                counts[color] += 1
                cumulative[color] += 1

    sheet_counts[sheet_name] = counts

for sheet_name, counts in sheet_counts.items():
    print(f"\n{sheet_name}")
    for color, count in counts.most_common():
        print(f"  {color}: {count}")

print("\nCUMULATIVE")
for color, count in cumulative.most_common():
    print(f"{color}: {count}")

## THREE HUMAN versus LLM

In [ ]:
import pandas as pd
from valid import versa
from sqlitedict import SqliteDict

config = versa.ExtractionConfig(df=pd.DataFrame(),
                        prompt_file="../prompts/extraction/iai.txt",
                        tool_calling=True,
                        llm="gpt-5-mini-2025-08-07",
                        response_schema={}, # type: ignore
                        n_replicates=3,
                        n_workers=200)

e = versa.Extraction(config)

PAIRS = {
    "TBI OLDER": "../outputs/extraction_combined_TBI_OLDER_apr_23.csv",
    "TBI YOUNGER": "../outputs/extraction_TBI_YOUNGER_apr_23.csv",
    "CSPINE": "../outputs/extraction_v2_CSPINE_apr_23.csv",
    "IAI": "../outputs/extraction_IAI_apr_23.csv"
}

DB_PAIRS = {
    "TBI OLDER": "../outputs/extraction_combined_TBI_OLDER_apr_23.sqlite",
    "TBI YOUNGER": "../outputs/extraction_TBI_YOUNGER_apr_23.sqlite",
    "CSPINE": "../outputs/extraction_v2_CSPINE_apr_23.sqlite",
    "IAI": "../outputs/extraction_IAI_apr_23.sqlite"
}

EXCEL_PATH = "../review_documents/PECARN THREE HUMAN CONSENSUS.xlsx"
xl = pd.ExcelFile(EXCEL_PATH)

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

LLM_OUTPUT_PATH = "../review_documents/PECARN HUMAN-LLM COOPERATION.xlsx"
RED_FILL = PatternFill(start_color="FF0000", end_color="FF0000", fill_type="solid")

def normalize_val(val):
    if pd.isna(val) or val == ".":
        return None
    try:
        return str(int(float(val)))
    except (ValueError, TypeError):
        return str(val).strip()

disagreement_map = {}
total_cells = 0
total_disagreements = 0
per_variable_data = []
per_sheet_data = []

df_llms = {}

with pd.ExcelWriter(LLM_OUTPUT_PATH, engine="openpyxl") as writer:
    for human_sheet, llm_path in PAIRS.items():
        df_human = pd.read_excel(xl, sheet_name=human_sheet + " HUMAN CONSENSUS")
        df_llm = pd.read_csv(llm_path)

        df_llm = df_llm[df_llm.EncounterKey.isin(set(df_human.EncounterKey.tolist()))]

        if "exclude" in df_human.columns:
            df_human = df_human[df_human["exclude"] != "yes"]
            df_llm = df_llm[df_llm.EncounterKey.isin(set(df_human.EncounterKey.tolist()))]
        
        df_llm_og = df_llm.copy()
        db = SqliteDict(DB_PAIRS[human_sheet], tablename="inference")

        def get_inference_column(ek: str):
            reps = [db.get(f"{ek}_rep{i}", None) for i in range(0,3)]
            reps = [json.loads(r) for r in reps if r]
            agg = e._aggregate_replicates(reps, remove_intermediates=False)
            return agg

        df_llm = df_llm[["EncounterKey"]].reset_index(drop=True)
        inference_df = pd.json_normalize(df_llm.EncounterKey.apply(get_inference_column))

        df_llm = pd.concat([df_llm, inference_df], axis=1).drop(columns=["risk_rom"], errors="ignore")

        risk_cols = [c for c in df_human.columns if c.startswith("risk_")]
        non_risk_cols = [c for c in df_human.columns if not c.startswith("risk_")]

        merged = df_human[non_risk_cols + risk_cols].merge(
            df_llm[["EncounterKey"] + risk_cols],
            on="EncounterKey",
            suffixes=("_human", "_llm"),
        )

        df_llms[human_sheet] = (df_llm[["EncounterKey"] + risk_cols].copy())

        output = merged[non_risk_cols].copy()
        discordant_cells = []

        sheet_n_cells = 0
        sheet_n_disagree = 0

        for col in risk_cols:
            col_h = f"{col}_human"
            col_l = f"{col}_llm"
            vals = []
            col_disagree = 0
            col_total = 0
            for idx, row in merged.iterrows():
                v_h = normalize_val(row[col_h])
                v_l = normalize_val(row[col_l])
                if v_h is None and v_l is None:
                    vals.append("")
                    continue
                col_total += 1
                if v_h == v_l:
                    vals.append(v_h if v_h is not None else "")
                else:
                    label = f"Human: {row[col_h]}, LLM: {row[col_l]}"
                    vals.append(label)
                    discordant_cells.append((len(vals) - 1, col))
                    col_disagree += 1
            output[col] = vals
            sheet_n_cells += col_total
            sheet_n_disagree += col_disagree
            per_variable_data.append({
                "dataset": human_sheet,
                "variable": col,
                "n": col_total,
                "disagreements": col_disagree,
                "proportion": round(col_disagree / col_total, 3) if col_total > 0 else 0,
            })

        total_cells += sheet_n_cells
        total_disagreements += sheet_n_disagree
        per_sheet_data.append({
            "dataset": human_sheet,
            "rated_cells": sheet_n_cells,
            "disagreements": sheet_n_disagree,
            "proportion": round(sheet_n_disagree / sheet_n_cells, 3) if sheet_n_cells > 0 else 0,
        })

        output.to_excel(writer, sheet_name=human_sheet, index=False)
        disagreement_map[human_sheet] = [
            (r, output.columns.get_loc(c)) for r, c in discordant_cells
        ]

# wb = load_workbook(LLM_OUTPUT_PATH)
# for sheet_name, cells in disagreement_map.items():
#     ws = wb[sheet_name]
#     for row_idx, col_idx in cells:
#         ws.cell(row=row_idx + 2, column=col_idx + 1).fill = RED_FILL

# wb.save(LLM_OUTPUT_PATH)
# print(f"Saved to {LLM_OUTPUT_PATH}")
# for sheet_name, cells in disagreement_map.items():
#     print(f"  {sheet_name}: {len(cells)} disagreement cells")

In [ ]:
import pandas as pd

# Example:
# dfs = {
#     "users": users_df,
#     "orders": orders_df,
#     "products": products_df
# }

with pd.ExcelWriter("../review_documents/LLM ALONE.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in df_llms.items():
        # Excel sheet names are limited to 31 characters
        df.to_excel(
            writer,
            sheet_name=str(sheet_name),
            index=False
        )

In [ ]:
per_sheet_data.append({
    "dataset": "TOTAL",
    "rated_cells": total_cells,
    "disagreements": total_disagreements,
    "proportion": round(total_disagreements / total_cells, 3) if total_cells > 0 else 0,
})

print("=== Per-Variable Discordance (Human vs LLM) ===")
display(pd.DataFrame(per_variable_data))

print("\n=== Per-Sheet Summary ===")
display(pd.DataFrame(per_sheet_data))

In [ ]:
LLM_OUTPUT_PATH

In [ ]:
import random

print(LLM_OUTPUT_PATH)
wb_check = load_workbook(LLM_OUTPUT_PATH)
checks_passed = 0
checks_total = 0

agree_checks = []
disagree_checks = []

for human_sheet, llm_path in PAIRS.items():
    ws = wb_check[human_sheet]

    df_h = pd.read_excel(xl, sheet_name=human_sheet + " HUMAN CONSENSUS")
    df_l = pd.read_csv(llm_path)
    df_l = df_l[df_l.EncounterKey.isin(set(df_h.EncounterKey.tolist()))]

    if "exclude" in df_h.columns:
        df_h = df_h[df_h["exclude"] != "yes"]
        df_l = df_l[df_l.EncounterKey.isin(set(df_h.EncounterKey.tolist()))]

    db = SqliteDict(DB_PAIRS[human_sheet], tablename="inference")

    def get_inference_column(ek: str):
        reps = [db.get(f"{ek}_rep{i}", None) for i in range(0, 3)]
        reps = [json.loads(r) for r in reps if r]
        agg = e._aggregate_replicates(reps, remove_intermediates=False)
        return agg

    df_l = df_l[["EncounterKey"]].reset_index(drop=True)
    inference_df = pd.json_normalize(df_l.EncounterKey.apply(get_inference_column))
    df_l = pd.concat([df_l, inference_df], axis=1).drop(columns=["risk_rom"], errors="ignore")

    risk_cols = [c for c in df_h.columns if c.startswith("risk_")]

    merged = df_h[["EncounterKey"] + risk_cols].merge(
        df_l[["EncounterKey"] + risk_cols],
        on="EncounterKey",
        suffixes=("_human", "_llm"),
    )

    headers = [ws.cell(row=1, column=c).value for c in range(1, ws.max_column + 1)]
    enc_col_idx = headers.index("EncounterKey") + 1

    encounter_rows = {}
    for row in range(2, ws.max_row + 1):
        encounter_rows[ws.cell(row=row, column=enc_col_idx).value] = row

    for _, mrow in merged.iterrows():
        enc_key = mrow["EncounterKey"]
        if enc_key not in encounter_rows:
            continue
        excel_row = encounter_rows[enc_key]
        for col in risk_cols:
            v_h = normalize_val(mrow[f"{col}_human"])
            v_l = normalize_val(mrow[f"{col}_llm"])
            if v_h is None and v_l is None:
                continue
            col_idx = headers.index(col) + 1
            cell = ws.cell(row=excel_row, column=col_idx)
            cell_val = str(cell.value) if cell.value is not None else ""
            has_red = cell.fill.start_color.rgb == "00FF0000"
            if v_h == v_l:
                agree_checks.append((human_sheet, enc_key, col, cell_val, has_red))
            else:
                disagree_checks.append((human_sheet, enc_key, col, cell_val, has_red))

random.seed(42)
sampled_agree = random.sample(agree_checks, min(50, len(agree_checks)))
sampled_disagree = random.sample(disagree_checks, min(50, len(disagree_checks)))

for sheet, enc, col, cell_val, has_red in sampled_agree:
    checks_total += 1
    ok = not has_red and "Human:" not in cell_val
    if ok:
        checks_passed += 1
    else:
        print(f"FAIL [agree] {sheet} | {enc} | {col}: val={cell_val}, red={has_red}")

for sheet, enc, col, cell_val, has_red in sampled_disagree:
    checks_total += 1
    ok = has_red and "Human:" in cell_val and "LLM:" in cell_val
    if ok:
        checks_passed += 1
    else:
        print(f"FAIL [disagree] {sheet} | {enc} | {col}: val={cell_val}, red={has_red}")

print(f"\n{checks_passed}/{checks_total} spot checks passed ({len(sampled_agree)} agreements, {len(sampled_disagree)} disagreements)")

## THREE HUMAN errors (corrections by LLM figure)

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles.colors import COLOR_INDEX
from collections import Counter
import xml.etree.ElementTree as ET

file_path = "../review_documents/PECARN HUMAN-LLM COOPERATION.xlsx"
wb = load_workbook(file_path)

THEME_ORDER = [
    "dk1", "lt1", "dk2", "lt2",
    "accent1", "accent2", "accent3", "accent4", "accent5", "accent6",
    "hlink", "folHlink"
]

def apply_tint(rgb, tint):
    if tint is None or tint == 0:
        return rgb.upper()

    r = int(rgb[0:2], 16)
    g = int(rgb[2:4], 16)
    b = int(rgb[4:6], 16)

    def tint_channel(c, tint_value):
        if tint_value < 0:
            return int(round(c * (1.0 + tint_value)))
        return int(round(c * (1.0 - tint_value) + 255 * tint_value))

    r = max(0, min(255, tint_channel(r, tint)))
    g = max(0, min(255, tint_channel(g, tint)))
    b = max(0, min(255, tint_channel(b, tint)))

    return f"{r:02X}{g:02X}{b:02X}"

def get_theme_map(workbook):
    theme_map = {}
    if not getattr(workbook, "loaded_theme", None):
        return theme_map

    root = ET.fromstring(workbook.loaded_theme)
    ns = {"a": "http://schemas.openxmlformats.org/drawingml/2006/main"}

    clr_scheme = root.find(".//a:clrScheme", ns)
    if clr_scheme is None:
        return theme_map

    for i, key in enumerate(THEME_ORDER):
        elem = clr_scheme.find(f"a:{key}", ns)
        if elem is None:
            continue

        child = list(elem)[0] if list(elem) else None
        if child is None:
            continue

        if child.tag.endswith("sysClr"):
            rgb = child.attrib.get("lastClr")
        else:
            rgb = child.attrib.get("val")

        if rgb:
            theme_map[i] = rgb.upper()

    return theme_map

THEME_MAP = get_theme_map(wb)

def resolve_color(color_obj):
    if color_obj is None:
        return None

    ctype = color_obj.type

    if ctype == "rgb" and color_obj.rgb:
        rgb = color_obj.rgb[-6:]
        return f"#{rgb.upper()}"

    if ctype == "indexed" and color_obj.indexed is not None:
        idx = int(color_obj.indexed)
        if 0 <= idx < len(COLOR_INDEX):
            rgb = COLOR_INDEX[idx]
            if rgb and len(rgb) >= 6:
                return f"#{rgb[-6:].upper()}"
        return f"indexed:{idx}"

    if ctype == "theme" and color_obj.theme is not None:
        theme_idx = int(color_obj.theme)
        base_rgb = THEME_MAP.get(theme_idx)
        if base_rgb:
            tint = getattr(color_obj, "tint", 0) or 0
            rgb = apply_tint(base_rgb, tint)
            return f"#{rgb.upper()}"
        return f"theme:{theme_idx}"

    if ctype == "auto":
        return "auto"

    return None

def get_fill_color(cell):
    fill = cell.fill
    if fill is None or fill.fill_type != "solid":
        return None
    return resolve_color(fill.fgColor)

sheet_counts = {}
cumulative = Counter()

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    counts = Counter()

    for row in ws.iter_rows():
        for cell in row:
            color = get_fill_color(cell)
            if color:
                counts[color] += 1
                cumulative[color] += 1

    sheet_counts[sheet_name] = counts

for sheet_name, counts in sheet_counts.items():
    print(f"\n{sheet_name}")
    for color, count in counts.most_common():
        print(f"  {color}: {count}")

print("\nCUMULATIVE")
for color, count in cumulative.most_common():
    print(f"{color}: {count}")

## HUMAN CONSENSUS vs THREE HUMAN CONSENSUS differences

In [ ]:
import pandas as pd

FILE_A = "../review_documents/PECARN HUMAN CONSENSUS.xlsx"
FILE_B = "../review_documents/PECARN THREE HUMAN CONSENSUS.xlsx"

SHEETS = [
    "TBI OLDER HUMAN CONSENSUS",
    "TBI YOUNGER HUMAN CONSENSUS",
    "CSPINE HUMAN CONSENSUS",
    "IAI HUMAN CONSENSUS",
]

def normalize_val(val):
    if pd.isna(val) or val == ".":
        return None
    try:
        return str(int(float(val)))
    except (ValueError, TypeError):
        return str(val).strip()

differences = []

for sheet in SHEETS:
    df_a = pd.read_excel(FILE_A, sheet_name=sheet)
    df_b = pd.read_excel(FILE_B, sheet_name=sheet)

    risk_cols_a = [c for c in df_a.columns if c.startswith("risk_")]
    risk_cols_b = [c for c in df_b.columns if c.startswith("risk_")]
    risk_cols = sorted(set(risk_cols_a) & set(risk_cols_b))

    merged = df_a[["EncounterKey"] + risk_cols].merge(
        df_b[["EncounterKey"] + risk_cols],
        on="EncounterKey",
        suffixes=("_a", "_b"),
    )

    for col in risk_cols:
        for _, row in merged.iterrows():
            v_a = normalize_val(row[f"{col}_a"])
            v_b = normalize_val(row[f"{col}_b"])
            if v_a != v_b:
                differences.append({
                    "sheet": sheet,
                    "EncounterKey": row["EncounterKey"],
                    "column": col,
                    "HUMAN_CONSENSUS": v_a,
                    "THREE_HUMAN_CONSENSUS": v_b,
                })

print(f"Total differences: {len(differences)}\n")
for d in differences:
    print(f"  {d['sheet']} | {d['EncounterKey']} | {d['column']} | {d['HUMAN_CONSENSUS']} -> {d['THREE_HUMAN_CONSENSUS']}")